# Hybrid Retrieval — Heavy Compute (Colab)

**What this notebook does:**
- Step 03: Build BM25 (Lucene) indexes for all 4 datasets
- Step 04: Encode corpora with BGE + build FAISS indexes
- Step 05: Run BM25 and Dense baselines, save TREC run files

**Dataset sizes used:**
| Dataset | Role | Corpus |
|---|---|---|
| MS MARCO | Training (fusion D & E) | **500K sampled** (from 8.84M) |
| TREC-COVID | Evaluation | 171K (full) |
| ArguAna | Evaluation | 8.6K (full) |
| FiQA | Evaluation | 57K (full) |



## Cell 1 — Check GPU & Disk

In [1]:
import subprocess, torch
print('=== GPU ===')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                '--format=csv,noheader'])
print(f'\nPyTorch CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('\n=== Disk ===')
subprocess.run(['df', '-h', '/'])

=== GPU ===

PyTorch CUDA: True
GPU: NVIDIA A100-SXM4-40GB

=== Disk ===


CompletedProcess(args=['df', '-h', '/'], returncode=0)

## Cell 2 — Install Java 21

In [2]:
%%bash
apt-get install -y -q openjdk-21-jdk 2>/dev/null
java -version
echo "JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))"

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common
  librsvg2-common libxcomposite1 libxt-dev libxtst6 libxxf86dga1
  openjdk-21-jdk-headless openjdk-21-jre openjdk-21-jre-headless
  session-migration x11-utils
Suggested packages:
  gvfs libxt-doc openjdk-21-demo openjdk-21-source visualvm libnss-mdns
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgtk-3-0 libgtk-3-bin libgtk-3-common
  librsv

openjdk version "21.0.10" 2026-01-20
OpenJDK Runtime Environment (build 21.0.10+7-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 21.0.10+7-Ubuntu-122.04, mixed mode, sharing)


## Cell 3 — Install Python packages

In [3]:
%%bash
pip install -q pyserini==0.22.1
pip install -q faiss-gpu
pip install -q sentence-transformers==2.7.0
pip install -q beir==2.0.0
pip install -q pytrec-eval-terrier
pip install -q nltk pyyaml tqdm
python -c "import nltk; nltk.download('stopwords', quiet=True)"
echo '✅ All packages installed.'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 133.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.2/314.2 kB 33.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 136.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.2/219.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


## Cell 4 — Mount Google Drive & create directories

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
from pathlib import Path

# ── Edit this if needed ──
PROJECT_ROOT = Path('/content/drive/MyDrive/hybrid_retrieval')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

for d in [
    'data/beir', 'data/cache',
    'indexes/bm25', 'indexes/dense',
    'runs/bm25', 'runs/dense',
    'models/trained', 'logs', 'report/tables',
]:
    (PROJECT_ROOT / d).mkdir(parents=True, exist_ok=True)

# Paths dict used throughout the notebook
PATHS = {
    'data':          str(PROJECT_ROOT / 'data/beir'),
    'cache':         str(PROJECT_ROOT / 'data/cache'),
    'indexes_bm25':  str(PROJECT_ROOT / 'indexes/bm25'),
    'indexes_dense': str(PROJECT_ROOT / 'indexes/dense'),
    'runs_bm25':     str(PROJECT_ROOT / 'runs/bm25'),
    'runs_dense':    str(PROJECT_ROOT / 'runs/dense'),
}
print(f'Project root: {PROJECT_ROOT}')
print('Directories ready ✅')

Mounted at /content/drive
Project root: /content/drive/MyDrive/hybrid_retrieval
Directories ready ✅


## Cell 5 — Global constants & helpers

In [13]:
import json, numpy as np, faiss, torch, time, subprocess, zipfile
from pathlib import Path
from beir.datasets.data_loader import GenericDataLoader
import pytrec_eval

QUERY_PREFIX     = 'Represent this sentence for searching relevant passages: '
MSMARCO_SAMPLE   = 500_000
TOP_K            = 1000
SEED             = 42

DATASETS = [
    ('arguana',    'test', 8674,     1406),
    ('fiqa',       'test', 57638,    648),
    ('trec-covid', 'test', 171332,   50),
    ('msmarco',    'dev',  MSMARCO_SAMPLE, 6980),
]

DATA_DIRS = {
    'arguana':    Path(PATHS['data']) / 'arguana',
    'fiqa':       Path(PATHS['data']) / 'fiqa',
    'trec-covid': Path(PATHS['data']) / 'trec-covid',
    'msmarco':    Path(PATHS['data']) / 'msmarco_sampled',
}

def write_trec(run, path, tag='retriever'):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        for qid, hits in run.items():
            for rank, (docid, score) in enumerate(hits[:TOP_K], 1):
                f.write(f'{qid} Q0 {docid} {rank} {score:.6f} {tag}\n')

def eval_run(run_dict, qrels_dict):
    evaluator = pytrec_eval.RelevanceEvaluator(
        qrels_dict, {'ndcg_cut_10', 'map', 'recall_100'})
    per_q = evaluator.evaluate(run_dict)
    return {m: round(sum(v[m] for v in per_q.values()) / len(per_q), 4)
            for m in ['ndcg_cut_10', 'map', 'recall_100']}

def to_pytrec(run):
    return {qid: {d: s for d, s in hits} for qid, hits in run.items()}

def qrels_to_pytrec(qrels):
    return {qid: {d: int(r) for d, r in docs.items()} for qid, docs in qrels.items()}

print('Constants & DATA_DIRS ready ✅')
for name, d in DATA_DIRS.items():
    exists = '✅' if (d / 'corpus.jsonl').exists() else '❌ corpus.jsonl missing'
    print(f'  {name}: {d}  {exists}')

Constants & DATA_DIRS ready ✅
  arguana: /content/drive/MyDrive/hybrid_retrieval/data/beir/arguana  ✅
  fiqa: /content/drive/MyDrive/hybrid_retrieval/data/beir/fiqa  ✅
  trec-covid: /content/drive/MyDrive/hybrid_retrieval/data/beir/trec-covid  ✅
  msmarco: /content/drive/MyDrive/hybrid_retrieval/data/beir/msmarco_sampled  ✅


## Cell 6 — Download datasets
MS MARCO is downloaded in full but we only **sample 500K docs** for indexing.  
All 3 evaluation datasets are used in full.

In [14]:
BASE_URL  = 'https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip'
data_root = Path(PATHS['data'])

for name, split, _, _ in DATASETS:
    corpus_file = data_root / name / 'corpus.jsonl'
    if corpus_file.exists():
        print(f'[{name}] Already exists — skipping.')
        continue
    print(f'[{name}] Downloading ...')
    zip_path = data_root / f'{name}.zip'
    subprocess.run([
        'wget', '-c', '--tries=5', '-q', '--show-progress',
        '-O', str(zip_path), BASE_URL.format(name=name)
    ], check=True)
    with zipfile.ZipFile(str(zip_path)) as z:
        z.extractall(str(data_root))
    corpus, queries, qrels = GenericDataLoader(str(data_root / name)).load(split=split)
    print(f'  corpus={len(corpus):,}  queries={len(queries):,} ✅')
    del corpus, queries, qrels

print('\n✅ All datasets downloaded.')

[arguana] Already exists — skipping.
[fiqa] Already exists — skipping.
[trec-covid] Already exists — skipping.
[msmarco] Already exists — skipping.

✅ All datasets downloaded.


## Cell 7 — Sample MS MARCO corpus to 500K
Streams `corpus.jsonl` directly — never loads 8.84M docs into RAM.  
All docs that appear in dev qrels are guaranteed to be included.

In [20]:
import random, json, shutil
random.seed(SEED)

msmarco_dir    = Path(PATHS['data']) / 'msmarco'
sampled_dir    = Path(PATHS['data']) / 'msmarco_sampled'
sampled_corpus = sampled_dir / 'corpus.jsonl'

# Check content rather than just file existence
if sampled_corpus.exists() and sampled_corpus.stat().st_size > 1000:
    n_lines = sum(1 for _ in open(sampled_corpus))
    print(f'Sampled corpus already exists ({n_lines:,} docs) — skipping.')
else:
    sampled_dir.mkdir(parents=True, exist_ok=True)
    # Delete empty file (the one causing the problem)
    if sampled_corpus.exists():
        sampled_corpus.unlink()
        print('Deleted empty corpus.jsonl, rebuilding ...')

    # Step 1: Read dev qrels to determine docids that must be kept
    print('Loading dev qrels ...')
    _, _, qrels_dev = GenericDataLoader(str(msmarco_dir)).load(split='dev')
    must_have = set()
    for docs in qrels_dev.values():
        must_have.update(docs.keys())
    print(f'  Must-include relevant docs: {len(must_have):,}')

    # Step 2: Stream corpus.jsonl, reservoir sampling
    print(f'Streaming corpus → sampling {MSMARCO_SAMPLE:,} docs ...')
    must_docs = {}   # docid → {'title': str, 'text': str}
    reservoir = []   # non-relevant candidate pool
    n_seen    = 0
    fill_target = MSMARCO_SAMPLE - len(must_have)

    with open(msmarco_dir / 'corpus.jsonl') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj   = json.loads(line)
            docid = obj.get('_id') or obj.get('id')
            title = obj.get('title', '')
            text  = obj.get('text', '')

            if docid in must_have:
                must_docs[docid] = {'title': title, 'text': text}
                continue

            # Reservoir sampling (Algorithm R)
            if len(reservoir) < fill_target:
                reservoir.append({'_id': docid, 'title': title, 'text': text})
            else:
                j = random.randint(0, n_seen)
                if j < fill_target:
                    reservoir[j] = {'_id': docid, 'title': title, 'text': text}
            n_seen += 1

            if n_seen % 500_000 == 0:
                print(f'  Scanned {n_seen/1e6:.1f}M non-relevant docs ...')

    random.shuffle(reservoir)
    fill_docs = reservoir[:fill_target]

    # Step 3: Write out sampled corpus
    print('Writing sampled corpus.jsonl ...')
    total = 0
    with open(sampled_corpus, 'w') as fout:
        # First write must-have relevant docs (this is the key fix: use dict instead of tuple)
        for docid, doc in must_docs.items():
            fout.write(json.dumps({
                '_id': docid,
                'title': doc['title'],
                'text':  doc['text']
            }) + '\n')
            total += 1
        # Then write random fill docs
        for doc in fill_docs:
            fout.write(json.dumps(doc) + '\n')
            total += 1

    # Copy queries/ and qrels/ (directly linked to original files)
    for subdir in ['queries', 'qrels']:
        src = msmarco_dir / subdir
        dst = sampled_dir / subdir
        if src.exists() and not dst.exists():
            shutil.copytree(str(src), str(dst))

    print(f'\nDone!')
    print(f'  Relevant docs:  {len(must_docs):,}')
    print(f'  Random fill:    {len(fill_docs):,}')
    print(f'  Total written:  {total:,}')


n_lines = sum(1 for _ in open(sampled_corpus))
size_mb = sampled_corpus.stat().st_size / 1e6
print(f'\nFinal check: {n_lines:,} lines, {size_mb:.1f} MB ✅')


Deleted empty corpus.jsonl, rebuilding ...
Loading dev qrels ...


  0%|          | 0/8841823 [00:00<?, ?it/s]

  Must-include relevant docs: 7,433
Streaming corpus → sampling 500,000 docs ...
  Scanned 0.5M non-relevant docs ...
  Scanned 1.0M non-relevant docs ...
  Scanned 1.5M non-relevant docs ...
  Scanned 2.0M non-relevant docs ...
  Scanned 2.5M non-relevant docs ...
  Scanned 3.0M non-relevant docs ...
  Scanned 3.5M non-relevant docs ...
  Scanned 4.0M non-relevant docs ...
  Scanned 4.5M non-relevant docs ...
  Scanned 5.0M non-relevant docs ...
  Scanned 5.5M non-relevant docs ...
  Scanned 6.0M non-relevant docs ...
  Scanned 6.5M non-relevant docs ...
  Scanned 7.0M non-relevant docs ...
  Scanned 7.5M non-relevant docs ...
  Scanned 8.0M non-relevant docs ...
  Scanned 8.5M non-relevant docs ...
Writing sampled corpus.jsonl ...

Done!
  Relevant docs:  7,433
  Random fill:    492,567
  Total written:  500,000

Final check: 500,000 lines, 193.7 MB ✅


## Cell 8 — Build BM25 indexes (~5 min total)

In [16]:
import json, sys, subprocess

# Map dataset name → actual data directory
DATA_DIRS = {
    'arguana':    Path(PATHS['data']) / 'arguana',
    'fiqa':       Path(PATHS['data']) / 'fiqa',
    'trec-covid': Path(PATHS['data']) / 'trec-covid',
    'msmarco':    Path(PATHS['data']) / 'msmarco_sampled',  # ← sampled version
}

def build_bm25_index(name, data_path, split):
    index_dir = Path(PATHS['indexes_bm25']) / name
    if (index_dir / 'index.properties').exists():
        print(f'[{name}] BM25 index exists — skipping.')
        return

    # Convert to Pyserini format
    pyserini_dir = data_path / 'pyserini_corpus'
    pyserini_dir.mkdir(exist_ok=True)
    out_file = pyserini_dir / 'corpus.jsonl'

    if not out_file.exists():
        # Stream directly from corpus.jsonl for memory efficiency
        print(f'[{name}] Converting corpus to Pyserini format ...')
        with open(data_path / 'corpus.jsonl') as fin, open(out_file, 'w') as fout:
            for line in fin:
                obj      = json.loads(line)
                docid    = obj.get('_id') or obj.get('id')
                title    = obj.get('title', '').strip()
                text     = obj.get('text',  '').strip()
                contents = (title + ' ' + text).strip() if title else text
                fout.write(json.dumps({'id': docid, 'contents': contents}) + '\n')

    index_dir.mkdir(parents=True, exist_ok=True)
    print(f'[{name}] Building Lucene index ...')
    r = subprocess.run([
        sys.executable, '-m', 'pyserini.index.lucene',
        '--collection', 'JsonCollection',
        '--input',      str(pyserini_dir),
        '--index',      str(index_dir),
        '--generator',  'DefaultLuceneDocumentGenerator',
        '--threads',    '4',
        '--storePositions', '--storeDocvectors', '--storeRaw',
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print('  ERROR:', r.stderr[-800:])
    else:
        print(f'  [{name}] BM25 index ✅')

for name, split, _, _ in DATASETS:
    build_bm25_index(name, DATA_DIRS[name], split)

print('\n✅ All BM25 indexes done.')

[arguana] Building Lucene index ...
  [arguana] BM25 index ✅
[fiqa] Building Lucene index ...
  [fiqa] BM25 index ✅
[trec-covid] Building Lucene index ...
  [trec-covid] BM25 index ✅
[msmarco] Building Lucene index ...
  [msmarco] BM25 index ✅

✅ All BM25 indexes done.


In [33]:
import shutil, subprocess, sys, json
from pathlib import Path

msmarco_sampled = Path(PATHS['data']) / 'msmarco_sampled'

# Delete empty index and pyserini_corpus
for target in [
    Path(PATHS['indexes_bm25']) / 'msmarco',
    msmarco_sampled / 'pyserini_corpus',
]:
    if target.exists():
        shutil.rmtree(str(target))
        print(f'Deleted: {target.name}')

# Convert to Pyserini format again
pyserini_dir = msmarco_sampled / 'pyserini_corpus'
pyserini_dir.mkdir(exist_ok=True)

print('Converting sampled corpus to Pyserini format ...')
count = 0
with open(msmarco_sampled / 'corpus.jsonl') as fin, \
     open(pyserini_dir / 'corpus.jsonl', 'w') as fout:
    for line in fin:
        obj      = json.loads(line)
        docid    = obj.get('_id') or obj.get('id')
        title    = obj.get('title', '').strip()
        text     = obj.get('text',  '').strip()
        contents = (title + ' ' + text).strip() if title else text
        fout.write(json.dumps({'id': docid, 'contents': contents}) + '\n')
        count += 1
print(f'Converted {count:,} docs ✅')

# Rebuild Lucene index
index_dir = Path(PATHS['indexes_bm25']) / 'msmarco'
index_dir.mkdir(parents=True, exist_ok=True)
print('Building BM25 index ...')
r = subprocess.run([
    sys.executable, '-m', 'pyserini.index.lucene',
    '--collection', 'JsonCollection',
    '--input',      str(pyserini_dir),
    '--index',      str(index_dir),
    '--generator',  'DefaultLuceneDocumentGenerator',
    '--threads',    '4',
    '--storePositions', '--storeDocvectors', '--storeRaw',
], capture_output=True, text=True)

if r.returncode != 0:
    print('ERROR:', r.stderr[-500:])
else:
    print('BM25 index rebuilt ✅')

Deleted: msmarco
Deleted: pyserini_corpus
Converting sampled corpus to Pyserini format ...
Converted 500,000 docs ✅
Building BM25 index ...
BM25 index rebuilt ✅


In [34]:
from pyserini.search.lucene import LuceneSearcher

old_run = Path(PATHS['runs_bm25']) / 'msmarco.trec'
if old_run.exists():
    old_run.unlink()
    print('Deleted old run file')

searcher = LuceneSearcher(str(Path(PATHS['indexes_bm25']) / 'msmarco'))
searcher.set_bm25(k1=0.9, b=0.4)

_, queries, qrels = GenericDataLoader(str(DATA_DIRS['msmarco'])).load(split='dev')
qids   = list(queries.keys())
qtexts = list(queries.values())

print(f'Searching {len(qids):,} queries ...')
raw = searcher.batch_search(qtexts, qids, k=TOP_K, threads=4)
run = {qid: [(h.docid, h.score) for h in hits] for qid, hits in raw.items()}
write_trec(run, old_run, tag='bm25')

m = eval_run(to_pytrec(run), qrels_to_pytrec(qrels))
print(f'NDCG@10={m["ndcg_cut_10"]}  MAP={m["map"]}  R@100={m["recall_100"]}')

Deleted old run file


  0%|          | 0/500000 [00:00<?, ?it/s]

Searching 6,980 queries ...
NDCG@10=0.548  MAP=0.5025  R@100=0.883


## Cell 9 — BGE Encoding + FAISS indexes
All 4 datasets encoded in a single pass — each fit in GPU memory.  


In [21]:
import numpy as np, torch, time, json, subprocess, sys
from sentence_transformers import SentenceTransformer

try:
    import faiss
except ImportError:
    print("Warning: faiss-gpu could not be imported. Attempting to install faiss-cpu as a fallback...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"])
        import faiss
        print("faiss-cpu installed and imported successfully.")
    except Exception as e:
        print(f"Error installing faiss-cpu: {e}")
        print("Please ensure faiss is installed in the environment for this cell to run.")
        raise # Re-raise to indicate failure if fallback also fails

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model  = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
print(f'Model loaded on {device}')

def encode_and_index(name, data_path):
    index_dir = Path(PATHS['indexes_dense']) / name
    if (index_dir / 'index.faiss').exists():
        print(f'[{name}] FAISS index exists — skipping.')
        return

    corpus_file = data_path / 'corpus.jsonl'


    if not corpus_file.exists():
        raise FileNotFoundError(
            f'[{name}] corpus.jsonl not found at {corpus_file}\n'
            f'  → If msmarco: re-run Cell 7 (sampling) first.'
        )

    print(f'[{name}] Reading corpus from {corpus_file} ...')
    docids, texts = [], []
    with open(corpus_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj   = json.loads(line)
            docid = obj.get('_id') or obj.get('id') or obj.get('docid')
            title = obj.get('title', '').strip()
            text  = obj.get('text',  '').strip()
            if docid:
                docids.append(docid)
                texts.append((title + ' ' + text).strip() if title else text)

    print(f'  {len(texts):,} docs loaded')


    if len(texts) == 0:
        raise ValueError(
            f'[{name}] 0 docs loaded from {corpus_file}\n'
            f'  → Check file is not empty: wc -l {corpus_file}'
        )

    t0  = time.time()
    emb = model.encode(
        texts,
        batch_size=512,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).astype('float32')
    print(f'  Encoded {emb.shape[0]:,} docs in {(time.time()-t0)/60:.1f} min')

    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)

    index_dir.mkdir(parents=True, exist_ok=True)
    faiss.write_index(index, str(index_dir / 'index.faiss'))
    np.save(str(index_dir / 'docids.npy'), np.array(docids))
    print(f'  [{name}] FAISS saved ({index.ntotal:,} vectors) ✅')
    del emb

for name, _, _, _ in DATASETS:
    encode_and_index(name, DATA_DIRS[name])

print('\n✅ All FAISS indexes done.')

Model loaded on cuda
[arguana] FAISS index exists — skipping.
[fiqa] FAISS index exists — skipping.
[trec-covid] FAISS index exists — skipping.
[msmarco] Reading corpus from /content/drive/MyDrive/hybrid_retrieval/data/beir/msmarco_sampled/corpus.jsonl ...
  500,000 docs loaded


Batches:   0%|          | 0/977 [00:00<?, ?it/s]

  Encoded 500,000 docs in 14.1 min
  [msmarco] FAISS saved (500,000 vectors) ✅

✅ All FAISS indexes done.


In [28]:
import shutil
shutil.copy2(
    str(Path(PATHS['data']) / 'msmarco' / 'queries.jsonl'),
    str(Path(PATHS['data']) / 'msmarco_sampled' / 'queries.jsonl')
)
print('queries.jsonl copied ✅')
print('Size:', round((Path(PATHS['data']) / 'msmarco_sampled' / 'queries.jsonl').stat().st_size / 1e6, 1), 'MB')

queries.jsonl copied ✅
Size: 40.4 MB


## Cell 10 — BM25 baseline runs (~5 min total)

In [35]:
from pyserini.search.lucene import LuceneSearcher

for name, split, _, _ in DATASETS:
    run_path = Path(PATHS['runs_bm25']) / f'{name}.trec'
    if run_path.exists():
        print(f'[{name}] BM25 run exists — skipping.')
        continue

    searcher = LuceneSearcher(str(Path(PATHS['indexes_bm25']) / name))
    searcher.set_bm25(k1=0.9, b=0.4)

    _, queries, qrels = GenericDataLoader(str(DATA_DIRS[name])).load(split=split)
    qids, qtexts = list(queries.keys()), list(queries.values())

    print(f'[{name}] BM25 searching {len(qids):,} queries ...')
    raw = searcher.batch_search(qtexts, qids, k=TOP_K, threads=4)
    run = {qid: [(h.docid, h.score) for h in hits] for qid, hits in raw.items()}
    write_trec(run, run_path, tag='bm25')

    m = eval_run(to_pytrec(run), qrels_to_pytrec(qrels))
    print(f'  NDCG@10={m["ndcg_cut_10"]}  MAP={m["map"]}  R@100={m["recall_100"]}  ✅')

print('\n✅ BM25 baselines done.')

[arguana] BM25 run exists — skipping.
[fiqa] BM25 run exists — skipping.
[trec-covid] BM25 run exists — skipping.
[msmarco] BM25 run exists — skipping.

✅ BM25 baselines done.


## Cell 11 — Dense baseline runs (~10 min total)

In [36]:
for name, split, _, _ in DATASETS:
    run_path = Path(PATHS['runs_dense']) / f'{name}.trec'
    if run_path.exists():
        print(f'[{name}] Dense run exists — skipping.')
        continue

    index_dir = Path(PATHS['indexes_dense']) / name
    index  = faiss.read_index(str(index_dir / 'index.faiss'))
    docids = np.load(str(index_dir / 'docids.npy'), allow_pickle=True).tolist()

    _, queries, qrels = GenericDataLoader(str(DATA_DIRS[name])).load(split=split)
    qids, qtexts = list(queries.keys()), list(queries.values())

    print(f'[{name}] Encoding {len(qids):,} queries ...')
    q_emb = model.encode(
        [QUERY_PREFIX + q for q in qtexts],
        batch_size=512, normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype('float32')

    print(f'  Searching FAISS top-{TOP_K} ...')
    scores, indices = index.search(q_emb, TOP_K)
    run = {
        qid: [(docids[indices[i, j]], float(scores[i, j]))
              for j in range(TOP_K) if indices[i, j] != -1]
        for i, qid in enumerate(qids)
    }
    write_trec(run, run_path, tag='dense')

    m = eval_run(to_pytrec(run), qrels_to_pytrec(qrels))
    print(f'  NDCG@10={m["ndcg_cut_10"]}  MAP={m["map"]}  R@100={m["recall_100"]}  ✅')

print('\n✅ Dense baselines done.')

[arguana] Dense run exists — skipping.
[fiqa] Dense run exists — skipping.
[trec-covid] Dense run exists — skipping.
[msmarco] Dense run exists — skipping.

✅ Dense baselines done.


## Cell 12 — Expected baseline numbers
Use these to sanity-check your results before downloading.

| Dataset | BM25 NDCG@10 | BGE NDCG@10 |
|---|---|---|
| arguana | ~0.41 | ~0.63 |
| fiqa | ~0.24 | ~0.40 |
| trec-covid | ~0.65 | ~0.77 |
| msmarco (500K) | ~0.19 | ~0.35 |

> MS MARCO numbers will be lower than full-corpus due to the 500K sample — expected and noted in the report.

## Cell 13 — Package & download run files

In [37]:
from google.colab import files

zip_out = '/content/run_files.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for retriever in ['bm25', 'dense']:
        run_dir = Path(PATHS[f'runs_{retriever}'])
        for f_path in sorted(run_dir.glob('*.trec')):
            arcname = f'runs/{retriever}/{f_path.name}'
            zf.write(str(f_path), arcname)
            size_mb = f_path.stat().st_size / 1e6
            print(f'  Added: {arcname}  ({size_mb:.1f} MB)')

total_mb = Path(zip_out).stat().st_size / 1e6
print(f'\nTotal zip size: {total_mb:.1f} MB')
files.download(zip_out)
print('✅ Download started.')

  Added: runs/bm25/arguana.trec  (127.4 MB)
  Added: runs/bm25/fiqa.trec  (21.1 MB)
  Added: runs/bm25/msmarco.trec  (250.7 MB)
  Added: runs/bm25/trec-covid.trec  (1.6 MB)
  Added: runs/dense/arguana.trec  (127.5 MB)
  Added: runs/dense/fiqa.trec  (21.8 MB)
  Added: runs/dense/msmarco.trec  (258.8 MB)
  Added: runs/dense/trec-covid.trec  (1.7 MB)

Total zip size: 193.9 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started.


## Cell 14 — Transfer back to Mac
```bash
cd "/Users/jay/Documents/information retrive/final project"
unzip ~/Downloads/run_files.zip
ls runs/bm25/ runs/dense/
# Expected (each folder): arguana.trec  fiqa.trec  msmarco.trec  trec-covid.trec
```
Then continue with **§4 — Fusion Strategies (A–E)** on Mac. All fusion code is CPU-only.